# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Accessing metadata attributes (not by subscript)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published on: {dataset.metadata.date_published}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their IDs
print("Available record sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}, type: {field.data_type})")
    print("")

# For demonstration: show records from the first record set by @id
if record_sets:
    example_record_set_id = record_sets[0].id
    print(f"\nExample records from record set '{example_record_set_id}':")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        pprint.pprint(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. All entities are referenced by their `@id`.

In [ ]:
# Extract data from each record set via their @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:  # Only create a DataFrame if data is present
        dataframes[rs_id] = pd.DataFrame(records)

# Display available DataFrames and preview columns for the main dataset
for rs_id, df in dataframes.items():
    print(f"\nRecord set: {rs_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply key processing: filtering, normalization, and grouping. All field references use their `@id`.

In [ ]:
# Example: EDA on the main tabular record set
from IPython.display import display
import numpy as np

# We'll identify the main tabular record set (with the richest data) by heuristics (most columns/rows)
main_rs_id = None
max_cols = 0
for rs_id, df in dataframes.items():
    if df.shape[1] > max_cols and df.shape[0] > 0:
        max_cols = df.shape[1]
        main_rs_id = rs_id

assert main_rs_id is not None, "No main record set was found."

main_df = dataframes[main_rs_id]
print(f"Selected main record set @id: {main_rs_id}")
print(f"Available columns (@id): {main_df.columns.tolist()}")

# Choose a numeric field by @id
# We'll pick a candidate numeric column (e.g., age, if present) based on column names
candidate_numeric_fields = [col for col in main_df.columns if 'age' in col.lower() or main_df[col].dtype in (np.int64, np.float64)]

# Fallback: select first numeric-ish column
numeric_field_id = candidate_numeric_fields[0] if candidate_numeric_fields else list(main_df.columns)[0]
print(f"Using numeric field: {numeric_field_id}")

# Filter for values above a threshold (arbitrary, e.g., >40 if Age, else >10)
try:
    threshold = 40 if 'age' in numeric_field_id.lower() else 10
    filtered_df = main_df[main_df[numeric_field_id].astype(float) > threshold]
except Exception:
    # Fallback: don't filter
    threshold = None
    filtered_df = main_df.copy()

if threshold is not None:
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
else:
    print(f"Could not apply threshold filtering for {numeric_field_id}, using all records.")
display(filtered_df.head())

# Normalize the numeric field
try:
    mean_val = filtered_df[numeric_field_id].astype(float).mean()
    std_val = filtered_df[numeric_field_id].astype(float).std()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - mean_val
    ) / std_val
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
except Exception as e:
    print(f"Normalization not possible for field {numeric_field_id}: {e}")

# Group data by a categorical field (if present)
group_fields = [col for col in main_df.columns if any(k in col.lower() for k in ('sex', 'anatomical', 'group', 'category', 'site'))]
group_field_id = group_fields[0] if group_fields else None

if group_field_id is not None and group_field_id in filtered_df.columns:
    try:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
    except Exception as e:
        print(f"Grouping not possible: {e}")
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All field references are by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field (by @id)
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field_id].astype(float), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field is present, show boxplot by group
if group_field_id:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id].astype(float))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded and explored the FAIR^2 dataset using `mlcroissant`, referencing all entities by their `@id` fields.
- The structure includes >1 record set; the main table organizes patient characteristics using clinical and molecular variables.
- We demonstrated filtering, normalization, and grouping of a numeric field, and visualized its data distribution.
- The Croissant ecosystem allows clean, structured access to FAIR biomedical data for analysis and reproducibility.

**Next steps might include deeper statistical modeling and domain-driven analyses using the annotated schema.**